<a href="https://colab.research.google.com/github/Dsolanki8120/Transformer-KV-caching/blob/main/Transformer_KV_caching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset and setup

In [ ]:
import math
import time
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: "".join(itos[int(i)] for i in x)

data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))

train_data = data[:split]
val_data = data[split:]

vocab_size = len(chars)
block_size = 256
batch_size = 32
n_embd = 256
n_head = 8
n_layer = 6
dropout = 0.1

def get_batch(split="train"):
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - block_size - 1, (batch_size,))

    x = torch.stack([source[i:i + block_size] for i in ix])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in ix])

    return x.to(device), y.to(device)

print("Characters:", vocab_size)
print("Train:", len(train_data))
print("Validation:", len(val_data))

# Attention and Transformer block

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        assert n_embd % n_head == 0

        self.head_dim = n_embd // n_head

        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, past_key_value=None, use_cache=False):
        B, T, C = x.shape

        q, k, v = self.qkv(x).split(n_embd, dim=2)

        q = q.view(B, T, n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, n_head, self.head_dim).transpose(1, 2)

        past_length = 0

        if past_key_value is not None:
            past_k, past_v = past_key_value
            past_length = past_k.size(2)
            k = torch.cat((past_k, k), dim=2)
            v = torch.cat((past_v, v), dim=2)

        present = (k, v) if use_cache else None

        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)

        q_pos = torch.arange(
            past_length,
            past_length + T,
            device=x.device
        ).unsqueeze(1)

        k_pos = torch.arange(
            k.size(2),
            device=x.device
        ).unsqueeze(0)

        mask = k_pos <= q_pos
        scores = scores.masked_fill(
            ~mask.view(1, 1, T, k.size(2)),
            float("-inf")
        )

        attn = F.softmax(scores, dim=-1)
        attn = self.attn_drop(attn)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.proj(out)

        return self.resid_drop(out), present


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention()
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP()

    def forward(self, x, past_key_value=None, use_cache=False):
        a, present = self.attn(
            self.ln1(x),
            past_key_value=past_key_value,
            use_cache=use_cache
        )
        x = x + a
        x = x + self.mlp(self.ln2(x))
        return x, present

# Decoder-only Transformer and training

In [ ]:
class DecoderOnlyTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)

        self.blocks = nn.ModuleList(
            [TransformerBlock() for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, input_ids, targets=None,
                past_key_values=None, use_cache=False):

        B, T = input_ids.shape

        past_length = 0

        if past_key_values is not None:
            past_length = past_key_values[0][0].size(2)

        if past_length + T > block_size:
            raise ValueError("Sequence is longer than block_size")

        pos = torch.arange(
            past_length,
            past_length + T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(pos)
        )

        presents = []

        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]

            x, present = block(
                x,
                past_key_value=past,
                use_cache=use_cache
            )

            if use_cache:
                presents.append(present)

        logits = self.lm_head(self.ln_f(x))

        loss = None

        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )

        if use_cache:
            return logits, loss, tuple(presents)

        return logits, loss


model = DecoderOnlyTransformer().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for step in range(1000):
    xb, yb = get_batch()

    _, loss = model(xb, targets=yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters()) / 1e6,
    "M"
)

# Generation with and without KV cache

In [ ]:
@torch.no_grad()
def generate_without_cache(input_ids, max_new_tokens=100, temperature=0.8):
    model.eval()
    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        context = generated[:, -block_size:]
        logits, _ = model(context)

        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)

        next_token = torch.multinomial(probs, 1)
        generated = torch.cat((generated, next_token), dim=1)

    return generated


@torch.no_grad()
def generate_with_cache(input_ids, max_new_tokens=100, temperature=0.8):
    model.eval()
    generated = input_ids.clone()

    logits, _, past_key_values = model(
        input_ids,
        use_cache=True
    )

    for _ in range(max_new_tokens):
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)

        next_token = torch.multinomial(probs, 1)
        generated = torch.cat((generated, next_token), dim=1)

        current_length = past_key_values[0][0].size(2)

        if current_length >= block_size:
            break

        logits, _, past_key_values = model(
            next_token,
            past_key_values=past_key_values,
            use_cache=True
        )

    return generated


prompt = "ROMEO:"
prompt_ids = torch.tensor(
    [encode(prompt)],
    dtype=torch.long,
    device=device
)

print("WITHOUT KV CACHE\n")
output1 = generate_without_cache(
    prompt_ids,
    max_new_tokens=150
)
print(decode(output1[0].tolist()))

print("\nWITH KV CACHE\n")
output2 = generate_with_cache(
    prompt_ids,
    max_new_tokens=150
)
print(decode(output2[0].tolist()))

# Benchmark computation/memory tradeoff and save files

In [ ]:
@torch.no_grad()
def benchmark(context_lengths, new_tokens=32, repeats=3):
    rows = []

    for context_length in context_lengths:
        input_ids = torch.randint(
            0,
            vocab_size,
            (1, context_length),
            device=device
        )

        no_cache_times = []
        cache_times = []

        no_cache_memory = []
        cache_memory = []

        for _ in range(repeats):
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
                torch.cuda.synchronize()

            start = time.perf_counter()

            generate_without_cache(
                input_ids,
                max_new_tokens=new_tokens,
                temperature=1.0
            )

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            no_cache_times.append(time.perf_counter() - start)

            if torch.cuda.is_available():
                no_cache_memory.append(
                    torch.cuda.max_memory_allocated() / 1024**2
                )

        for _ in range(repeats):
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
                torch.cuda.synchronize()

            start = time.perf_counter()

            generate_with_cache(
                input_ids,
                max_new_tokens=new_tokens,
                temperature=1.0
            )

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            cache_times.append(time.perf_counter() - start)

            if torch.cuda.is_available():
                cache_memory.append(
                    torch.cuda.max_memory_allocated() / 1024**2
                )

        no_cache_time = np.mean(no_cache_times)
        cache_time = np.mean(cache_times)

        head_dim = n_embd // n_head

        kv_memory = (
            2
            * 1
            * context_length
            * n_layer
            * n_head
            * head_dim
            * 4
            / 1024**2
        )

        rows.append({
            "context_length": context_length,
            "no_cache_time_sec": no_cache_time,
            "cache_time_sec": cache_time,
            "speedup": no_cache_time / cache_time,
            "no_cache_peak_memory_MB": (
                np.mean(no_cache_memory)
                if no_cache_memory else np.nan
            ),
            "cache_peak_memory_MB": (
                np.mean(cache_memory)
                if cache_memory else np.nan
            ),
            "theoretical_kv_cache_MB": kv_memory
        })

    return pd.DataFrame(rows)


results = benchmark(
    [16, 32, 64, 128, 192],
    new_tokens=32,
    repeats=3
)

display(results.round(4))

plt.figure(figsize=(8, 5))
plt.plot(
    results.context_length,
    results.no_cache_time_sec,
    marker="o",
    label="Without KV Cache"
)
plt.plot(
    results.context_length,
    results.cache_time_sec,
    marker="o",
    label="With KV Cache"
)
plt.xlabel("Context Length")
plt.ylabel("Generation Time (s)")
plt.title("KV Cache vs No KV Cache")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    results.context_length,
    results.speedup,
    marker="o"
)
plt.xlabel("Context Length")
plt.ylabel("Speedup")
plt.title("KV Cache Speedup")
plt.grid()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    results.context_length,
    results.theoretical_kv_cache_MB,
    marker="o"
)
plt.xlabel("Context Length")
plt.ylabel("KV Cache Memory (MB)")
plt.title("KV Cache Memory Growth")
plt.grid()
plt.show()

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "stoi": stoi,
        "itos": itos,
        "config": {
            "vocab_size": vocab_size,
            "block_size": block_size,
            "n_embd": n_embd,
            "n_head": n_head,
            "n_layer": n_layer
        }
    },
    "decoder_transformer_kv_cache.pt"
)

results.to_csv("kv_cache_benchmark.csv", index=False)

print("Saved model: decoder_transformer_kv_cache.pt")
print("Saved benchmark: kv_cache_benchmark.csv")